In [1]:
from pathlib import Path

# ✅ Auto-detect repo root: graphCeylon-API/
REPO_ROOT = Path.cwd()
# if you're running from inside a subfolder, uncomment and adjust:
# REPO_ROOT = Path.cwd().parents[0]   # or parents[1], etc.

INPUT_CSV  = REPO_ROOT / "data/recommender_service/clean/sltda_arrivals_monthly.csv"
OUTPUT_ARIMA_CSV   = REPO_ROOT / "data/recommender_service/clean/sltda_arrivals_arima_forecast.csv"
OUTPUT_PROPHET_CSV = REPO_ROOT / "data/recommender_service/clean/sltda_arrivals_prophet_forecast.csv"
MODELS_DIR = REPO_ROOT / "services/recommender_service/models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("INPUT_CSV exists:", INPUT_CSV.exists(), INPUT_CSV)


REPO_ROOT: /Users/dev/Desktop/graphCeylon-API/models/recommender_service
INPUT_CSV exists: False /Users/dev/Desktop/graphCeylon-API/models/recommender_service/data/recommender_service/clean/sltda_arrivals_monthly.csv


In [5]:
# ARIMA Arrivals Forecasting - Notebook Version
import os


import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pandas.tseries.offsets import MonthBegin
from statsmodels.tsa.arima.model import ARIMA

# Paths
from pathlib import Path
INPUT_CSV = Path("/Users/dev/Desktop/graphCeylon-API/data/recommender_service/clean/sltda_arrivals_monthly.csv")
print("exists:", INPUT_CSV.exists())

OUTPUT_CSV = OUTPUT_ARIMA_CSV


exists: True


In [6]:
def load_arrivals() -> pd.DataFrame:
    df = pd.read_csv(INPUT_CSV, parse_dates=["date"])
    df = df.sort_values("date")
    df = df.set_index("date")

    # Ensure DatetimeIndex with monthly-start freq
    df.index = pd.to_datetime(df.index)
    df = df.asfreq("MS")   # 👈 Monthly Start frequency

    df["total_arrivals"] = df["total_arrivals"].astype(float)
    df.loc[df["total_arrivals"] < 1000, "total_arrivals"] = np.nan
    df["total_arrivals"] = df["total_arrivals"].interpolate(method="linear")

    print(f"Loaded {len(df)} monthly points from {df.index.min().date()} to {df.index.max().date()}")
    return df

df_arrivals = load_arrivals()


Loaded 94 monthly points from 2018-01-01 to 2025-10-01


In [7]:
def run_arima_forecast(df: pd.DataFrame, forecast_months: int = 12) -> pd.DataFrame:
    """
    Fit ARIMA baseline and forecast N future months.
    We manually build the future date index so statsmodels'
    lack-of-frequency warnings don't break us.
    """
    series = df["total_arrivals"]

    # Simple baseline model; you can later tune (p,d,q)
    model = ARIMA(series, order=(1, 1, 1))
    fitted = model.fit()

    # Get numeric forecast values
    forecast_values = fitted.forecast(steps=forecast_months)

    # Build proper monthly datetime index for the forecast
    last_date = df.index.max()
    start_future = last_date + MonthBegin(1)  # next month start
    future_index = pd.date_range(start=start_future, periods=forecast_months, freq="MS")

    future_df = pd.DataFrame(
        {
            "date": future_index,
            "forecast_total_arrivals": forecast_values.values,
        }
    )

    return future_df


In [8]:
def evaluate_arima(df: pd.DataFrame, val_months: int = 12):
    series = df["total_arrivals"]

    if len(series) <= val_months + 5:
        print("[ARIMA] Not enough data for a separate validation split; skipping metrics.")
        return None, None, None

    train = series.iloc[:-val_months]
    val   = series.iloc[-val_months:]

    model = ARIMA(train, order=(1, 1, 1))
    fitted = model.fit()

    forecast_vals = fitted.forecast(steps=val_months)

    y_true = val.values
    y_pred = forecast_vals.values

    mse = np.mean((y_true - y_pred)**2)
    rmse = math.sqrt(mse)
    mae = np.mean(np.abs(y_true - y_pred))

    # Robust date handling
    start_date = pd.to_datetime(val.index.min()).date()
    end_date   = pd.to_datetime(val.index.max()).date()

    print(f"[ARIMA] Validation window: {start_date} → {end_date}")
    print(f"[ARIMA] Validation RMSE: {rmse:.2f}")
    print(f"[ARIMA] Validation MAE : {mae:.2f}")

    return rmse, mae, fitted


In [9]:
rmse, mae, baseline_arima_model = evaluate_arima(df_arrivals, val_months=12)

if baseline_arima_model is not None:
    MODELS_DIR = Path("models")
    MODELS_DIR.mkdir(exist_ok=True)

    baseline_arima_path = MODELS_DIR / "arima_arrivals_1_1_1.pkl"

    with open(baseline_arima_path, "wb") as f:
        pickle.dump(baseline_arima_model, f)

    print("Saved baseline ARIMA(1,1,1) →", baseline_arima_path)


[ARIMA] Validation window: 2024-11-01 → 2025-10-01
[ARIMA] Validation RMSE: 134105.74
[ARIMA] Validation MAE : 110370.64


NameError: name 'pickle' is not defined

In [ ]:
def evaluate_arima_order(series: pd.Series, order, val_months: int = 12):
    """
    Evaluate a single ARIMA(order) on last `val_months` points.
    Returns (rmse, mae).
    """
    if len(series) <= val_months + 5:
        print("[ARIMA] Not enough data for validation split.")
        return None, None

    train = series.iloc[:-val_months]
    val = series.iloc[-val_months:]

    try:
        model = ARIMA(train, order=order)
        fitted = model.fit()
        forecast_vals = fitted.forecast(steps=val_months)

        y_true = val.values
        y_pred = forecast_vals.values

        mse = np.mean((y_true - y_pred) ** 2)
        rmse = math.sqrt(mse)
        mae = np.mean(np.abs(y_true - y_pred))
        return rmse, mae
    except Exception as e:
        # Some (p,d,q) combos can fail; just skip them
        # print(f"Failed for order={order}: {e}")
        return None, None


In [ ]:
series = df_arrivals["total_arrivals"]

p_values = [0, 1, 2, 3]
d_values = [0, 1, 2]
q_values = [0, 1, 2, 3]

best_order = None
best_rmse = float("inf")
best_mae = float("inf")

for p in p_values:
    for d in d_values:
        for q in q_values:
            order = (p, d, q)
            rmse, mae = evaluate_arima_order(series, order, val_months=12)
            if rmse is None:
                continue
            if rmse < best_rmse:
                best_rmse = rmse
                best_mae = mae
                best_order = order
                print(f"[ARIMA] New best order={order} RMSE={rmse:.2f} MAE={mae:.2f}")

print("\n=== Best ARIMA model on last 12 months ===")
print("order:", best_order)
print(f"RMSE : {best_rmse:.2f}")
print(f"MAE  : {best_mae:.2f}")


[ARIMA] New best order=(0, 0, 0) RMSE=229628.62 MAE=215211.43
[ARIMA] New best order=(0, 0, 1) RMSE=226760.21 MAE=211442.57
[ARIMA] New best order=(0, 0, 2) RMSE=217229.76 MAE=202196.03
[ARIMA] New best order=(0, 0, 3) RMSE=209985.59 MAE=196512.08
[ARIMA] New best order=(0, 1, 0) RMSE=140486.26 MAE=116421.50


e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


[ARIMA] New best order=(0, 1, 1) RMSE=133631.86 MAE=110011.33
[ARIMA] New best order=(0, 1, 3) RMSE=115609.51 MAE=94698.68


e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


[ARIMA] New best order=(1, 1, 3) RMSE=114632.16 MAE=94331.32


...

e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


[ARIMA] New best order=(2, 2, 2) RMSE=101822.72 MAE=84875.06
[ARIMA] New best order=(2, 2, 3) RMSE=95604.26 MAE=78085.91


e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'



=== Best ARIMA model on last 12 months ===
order: (2, 2, 3)
RMSE : 95604.26
MAE  : 78085.91


e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [10]:
best_order = (3, 2, 3)

def evaluate_fixed_arima(series: pd.Series, order, val_months: int = 12):
    train = series.iloc[:-val_months]
    val = series.iloc[-val_months:]

    model = ARIMA(train, order=order)
    fitted = model.fit()
    forecast_vals = fitted.forecast(steps=val_months)

    y_true = val.values
    y_pred = forecast_vals.values

    mse = np.mean((y_true - y_pred) ** 2)
    rmse = math.sqrt(mse)
    mae = np.mean(np.abs(y_true - y_pred))

    print(f"[ARIMA {order}] Validation window: {val.index.min().date()} → {val.index.max().date()}")
    print(f"[ARIMA {order}] Validation RMSE: {rmse:.2f}")
    print(f"[ARIMA {order}] Validation MAE : {mae:.2f}")

    return rmse, mae, fitted   # ✅ RETURN MODEL TOO


In [11]:
series = df_arrivals["total_arrivals"]
rmse_best, mae_best, arima_323_model = evaluate_fixed_arima(series, best_order, val_months=12)


[ARIMA (3, 2, 3)] Validation window: 2024-11-01 → 2025-10-01
[ARIMA (3, 2, 3)] Validation RMSE: 96860.00
[ARIMA (3, 2, 3)] Validation MAE : 76869.45


/opt/anaconda3/lib/python3.12/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [12]:
import pickle
from pathlib import Path

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

save_path = MODELS_DIR / "arima_arrivals_3_2_3.pkl"

with open(save_path, "wb") as f:
    pickle.dump(arima_323_model, f)

print("Saved tuned ARIMA(3,2,3) →", save_path)


Saved tuned ARIMA(3,2,3) → models/arima_arrivals_3_2_3.pkl


In [13]:
def run_arima_forecast_with_order(df: pd.DataFrame, order, forecast_months: int = 12) -> pd.DataFrame:
    series = df["total_arrivals"]
    model = ARIMA(series, order=order)
    fitted = model.fit()
    forecast_values = fitted.forecast(steps=forecast_months)

    last_date = df.index.max()
    start_future = last_date + MonthBegin(1)
    future_index = pd.date_range(start=start_future, periods=forecast_months, freq="MS")

    future_df = pd.DataFrame({
        "date": future_index,
        "forecast_total_arrivals": forecast_values.values,
    })
    return future_df

future_df_best = run_arima_forecast_with_order(df_arrivals, best_order, forecast_months=12)
future_df_best.head()


/opt/anaconda3/lib/python3.12/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/opt/anaconda3/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


,date,forecast_total_arrivals
0,2025-11-01,378165.911782
1,2025-12-01,414172.492121
2,2026-01-01,382874.407126
3,2026-02-01,362864.860979
4,2026-03-01,392279.334756


In [14]:
OUTPUT_CSV = Path("data_clean/forecast/sltda_arrivals_arima_forecast.csv")
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
future_df_best.to_csv(OUTPUT_CSV, index=False)
print("Saved best ARIMA forecast →", OUTPUT_CSV)


Saved best ARIMA forecast → data_clean/forecast/sltda_arrivals_arima_forecast.csv


In [15]:
tuned_arima_path = MODELS_DIR / "arima_arrivals_3_2_3.pkl"

with open(tuned_arima_path, "wb") as f:
    pickle.dump(best_arima_model, f)

print("Saved tuned ARIMA(3,2,3) →", tuned_arima_path)


NameError: name 'best_arima_model' is not defined

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

def evaluate_sarima(series: pd.Series,
                    val_months: int = 12,
                    p_values=(0, 1, 2),
                    d_values=(0, 1),
                    q_values=(0, 1, 2),
                    P_values=(0, 1, 2),
                    D_values=(0, 1),
                    Q_values=(0, 1, 2),
                    seasonal_periods=12):
    """
    Simple grid search over a small SARIMA space.
    Returns (best_order, best_seasonal_order, best_rmse, best_mae, best_model).
    """
    if len(series) <= val_months + 5:
        print("[SARIMA] Not enough data for validation split.")
        return None, None, None, None, None

    train = series.iloc[:-val_months]
    val = series.iloc[-val_months:]

    best_rmse = float("inf")
    best_mae = float("inf")
    best_order = None
    best_seasonal = None
    best_model = None

    for p in p_values:
        for d in d_values:
            for q in q_values:
                for P in P_values:
                    for D in D_values:
                        for Q in Q_values:
                            order = (p, d, q)
                            seasonal_order = (P, D, Q, seasonal_periods)
                            try:
                                model = SARIMAX(
                                    train,
                                    order=order,
                                    seasonal_order=seasonal_order,
                                    enforce_stationarity=False,
                                    enforce_invertibility=False,
                                )
                                fitted = model.fit(disp=False)
                                forecast_vals = fitted.forecast(steps=val_months)

                                y_true = val.values
                                y_pred = forecast_vals.values
                                mse = np.mean((y_true - y_pred) ** 2)
                                rmse = math.sqrt(mse)
                                mae = np.mean(np.abs(y_true - y_pred))

                                if rmse < best_rmse:
                                    best_rmse = rmse
                                    best_mae = mae
                                    best_order = order
                                    best_seasonal = seasonal_order
                                    best_model = fitted
                                    print(
                                        f"[SARIMA] New best {order} x {seasonal_order} "
                                        f"RMSE={rmse:.2f} MAE={mae:.2f}"
                                    )
                            except Exception as e:
                                # Some combos are unstable; just skip
                                # print(f"Failed for {order} x {seasonal_order}: {e}")
                                continue

    print("\n=== Best SARIMA model ===")
    print("order         :", best_order)
    print("seasonal order:", best_seasonal)
    print(f"RMSE          : {best_rmse:.2f}")
    print(f"MAE           : {best_mae:.2f}")

    return best_order, best_seasonal, best_rmse, best_mae, best_model

series = df_arrivals["total_arrivals"]
best_order, best_seasonal, best_rmse, best_mae, best_model = evaluate_sarima(series, val_months=12)


[SARIMA] New best (0, 0, 0) x (0, 0, 0, 12) RMSE=395433.65 MAE=387239.50
[SARIMA] New best (0, 0, 0) x (0, 0, 1, 12) RMSE=233524.14 MAE=230672.06


e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'


[SARIMA] New best (0, 0, 0) x (0, 1, 0, 12) RMSE=59444.36 MAE=56812.33


e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'


[SARIMA] New best (0, 0, 0) x (1, 1, 0, 12) RMSE=57374.69 MAE=54733.57


...

e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters ex

[SARIMA] New best (2, 1, 0) x (2, 1, 1, 12) RMSE=16737.75 MAE=15001.57


e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations


=== Best SARIMA model ===
order         : (2, 1, 0)
seasonal order: (2, 1, 1, 12)
RMSE          : 16737.75
MAE           : 15001.57


e:\Research_Final\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [ ]:
import warnings
from statsmodels.tsa.statespace.sarimax import SARIMAX

def evaluate_sarima_refined(series: pd.Series,
                            val_months: int = 12):
    """
    Refined SARIMA search around the previously found good region.
    Returns (best_order, best_seasonal_order, best_rmse, best_mae, best_model).
    """
    if len(series) <= val_months + 5:
        print("[SARIMA] Not enough data for validation split.")
        return None, None, None, None, None

    train = series.iloc[:-val_months]
    val = series.iloc[-val_months:]

    # 🔍 Focused search space around (2,1,0)x(2,1,1,12)
    p_values = [1, 2, 3]
    d_values = [1]          # you already differenced once
    q_values = [0, 1, 2]

    P_values = [1, 2]
    D_values = [1]
    Q_values = [0, 1, 2]
    seasonal_periods = 12

    best_rmse = float("inf")
    best_mae = float("inf")
    best_order = None
    best_seasonal = None
    best_model = None

    for p in p_values:
        for d in d_values:
            for q in q_values:
                for P in P_values:
                    for D in D_values:
                        for Q in Q_values:
                            order = (p, d, q)
                            seasonal_order = (P, D, Q, seasonal_periods)
                            try:
                                with warnings.catch_warnings():
                                    warnings.filterwarnings("ignore")
                                    model = SARIMAX(
                                        train,
                                        order=order,
                                        seasonal_order=seasonal_order,
                                        enforce_stationarity=False,
                                        enforce_invertibility=False,
                                    )
                                    fitted = model.fit(disp=False, maxiter=200)

                                forecast_vals = fitted.forecast(steps=val_months)

                                y_true = val.values
                                y_pred = forecast_vals.values
                                mse = np.mean((y_true - y_pred) ** 2)
                                rmse = math.sqrt(mse)
                                mae = np.mean(np.abs(y_true - y_pred))

                                if rmse < best_rmse:
                                    best_rmse = rmse
                                    best_mae = mae
                                    best_order = order
                                    best_seasonal = seasonal_order
                                    best_model = fitted
                                    print(
                                        f"[SARIMA refined] New best {order} x {seasonal_order} "
                                        f"RMSE={rmse:.2f} MAE={mae:.2f}"
                                    )
                            except Exception:
                                continue

    print("\n=== Best SARIMA (refined) model ===")
    print("order         :", best_order)
    print("seasonal order:", best_seasonal)
    print(f"RMSE          : {best_rmse:.2f}")
    print(f"MAE           : {best_mae:.2f}")

    return best_order, best_seasonal, best_rmse, best_mae, best_model


In [ ]:
series = df_arrivals["total_arrivals"]
best_order_ref, best_seasonal_ref, best_rmse_ref, best_mae_ref, best_model_ref = evaluate_sarima_refined(series, val_months=12)


[SARIMA refined] New best (1, 1, 0) x (1, 1, 0, 12) RMSE=25351.46 MAE=21864.57
[SARIMA refined] New best (1, 1, 0) x (1, 1, 1, 12) RMSE=24396.68 MAE=19117.56
[SARIMA refined] New best (1, 1, 1) x (2, 1, 2, 12) RMSE=24087.11 MAE=20338.54
[SARIMA refined] New best (2, 1, 0) x (2, 1, 0, 12) RMSE=18522.97 MAE=16106.90
[SARIMA refined] New best (2, 1, 0) x (2, 1, 1, 12) RMSE=16737.75 MAE=15001.57

=== Best SARIMA (refined) model ===
order         : (2, 1, 0)
seasonal order: (2, 1, 1, 12)
RMSE          : 16737.75
MAE           : 15001.57


In [ ]:
import os
import pickle

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

sarima_path = MODELS_DIR / "sarima_arrivals_2_1_0_2_1_1_12.pkl"

with open(sarima_path, "wb") as f:
    pickle.dump(best_model_ref, f)

print("Saved SARIMA model to:", sarima_path)


Saved SARIMA model to: models\sarima_arrivals_2_1_0_2_1_1_12.pkl


LSTM

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import math


In [ ]:
series = df_arrivals["total_arrivals"].values.astype(float).reshape(-1, 1)

scaler = MinMaxScaler(feature_range=(0, 1))
series_scaled = scaler.fit_transform(series)  # shape (N, 1)


In [ ]:
WINDOW_SIZE = 24
VAL_MONTHS = 12

def create_sequences(data, window_size):
    """
    data: numpy array shape (N, 1) scaled
    returns X (num_samples, window_size, 1), y (num_samples, 1)
    """
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size])
    return np.array(X), np.array(y)

X_all, y_all = create_sequences(series_scaled, WINDOW_SIZE)
print("X_all shape:", X_all.shape, "y_all shape:", y_all.shape)


X_all shape: (70, 24, 1) y_all shape: (70, 1)


In [ ]:
# y_all length = len(series_scaled) - WINDOW_SIZE
# we want last VAL_MONTHS targets as validation
train_size = X_all.shape[0] - VAL_MONTHS

X_train = X_all[:train_size]
y_train = y_all[:train_size]
X_val   = X_all[train_size:]
y_val   = y_all[train_size:]

print("Train samples:", X_train.shape[0], "Val samples:", X_val.shape[0])


Train samples: 58 Val samples: 12


In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = TimeSeriesDataset(X_train, y_train)
val_dataset   = TimeSeriesDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)


In [ ]:
import torch
import torch.nn as nn

# -------------------------------
# DEVICE SETUP
# -------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [ ]:
class ArrivalLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=16, num_layers=1, dropout=0.0):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        last_out = out[:, -1, :]
        out = self.fc(last_out)
        return out

model = ArrivalLSTM(input_size=1, hidden_size=16, num_layers=1).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)  # slightly smaller LR


In [ ]:
EPOCHS = 100
best_val_rmse = float("inf")
best_state_dict = None
patience = 15
no_improve = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_losses = []

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    # validation
    model.eval()
    val_losses = []
    y_true_list = []
    y_pred_list = []

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            val_losses.append(loss.item())

            y_true_list.append(y_batch.cpu().numpy())
            y_pred_list.append(outputs.cpu().numpy())

    y_true_val = np.concatenate(y_true_list, axis=0)
    y_pred_val = np.concatenate(y_pred_list, axis=0)

    val_rmse = rmse_from_scaled(y_true_val, y_pred_val)
    val_mae  = mae_from_scaled(y_true_val, y_pred_val)

    # 🔥 early stopping logic
    if val_rmse < best_val_rmse - 1e-3:  # small tolerance
        best_val_rmse = val_rmse
        best_state_dict = model.state_dict()
        no_improve = 0
    else:
        no_improve += 1

    if epoch % 10 == 0 or epoch == 1:
        print(
            f"Epoch {epoch}/{EPOCHS} "
            f"- train_loss={np.mean(train_losses):.6f} "
            f"- val_loss={np.mean(val_losses):.6f} "
            f"- val_RMSE={val_rmse:.2f} "
            f"- val_MAE={val_mae:.2f}"
        )

    if no_improve >= patience:
        print(f"Early stopping at epoch {epoch} (no improvement for {patience} epochs).")
        break

# restore best weights
if best_state_dict is not None:
    model.load_state_dict(best_state_dict)
    print(f"Loaded best LSTM weights with val_RMSE={best_val_rmse:.2f}")


NameError: name 'rmse_from_scaled' is not defined

Prophet

In [ ]:
from pathlib import Path
import math

import numpy as np
import pandas as pd
from prophet import Prophet  # make sure 'prophet' is installed
from sklearn.metrics import mean_squared_error, mean_absolute_error

INPUT_CSV = Path("data_clean/forecast/sltda_arrivals_monthly.csv")
OUTPUT_CSV = Path("data_clean/forecast/sltda_arrivals_prophet_forecast.csv")
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def load_arrivals_for_prophet(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=["date"])
    df = df.sort_values("date")

    # Handle COVID collapse same as ARIMA
    df["total_arrivals"] = df["total_arrivals"].astype(float)
    df.loc[df["total_arrivals"] < 1000, "total_arrivals"] = np.nan
    df["total_arrivals"] = df["total_arrivals"].interpolate(method="linear")

    print(
        f"Loaded {len(df)} monthly points from "
        f"{df['date'].min().date()} to {df['date'].max().date()}"
    )
    return df

df_arrivals = load_arrivals_for_prophet(INPUT_CSV)
df_arrivals.head()


Loaded 94 monthly points from 2018-01-01 to 2025-10-01


,date,year,month,month_name,total_arrivals
0,2018-01-01,2018,1,Jan,152377.0
1,2018-01-02,2018,2,Feb,158782.0
2,2018-01-03,2018,3,Mar,160784.0
3,2018-01-04,2018,4,Apr,128013.0
4,2018-01-05,2018,5,May,95794.0


In [ ]:
VAL_MONTHS = 12

# Prophet expects columns 'ds' (date) and 'y' (target)
df_prophet = df_arrivals[["date", "total_arrivals"]].rename(
    columns={"date": "ds", "total_arrivals": "y"}
)

# Last 12 months as validation
train_df = df_prophet.iloc[:-VAL_MONTHS].copy()
val_df   = df_prophet.iloc[-VAL_MONTHS:].copy()

print("Train period:", train_df["ds"].min().date(), "→", train_df["ds"].max().date())
print("Val period  :", val_df["ds"].min().date(),   "→", val_df["ds"].max().date())


Train period: 2018-01-01 → 2024-10-01
Val period  : 2024-11-01 → 2025-10-01


In [ ]:
def fit_prophet_model(train_df: pd.DataFrame) -> Prophet:
    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode="additive",   # you can try "multiplicative" later
    )
    m.fit(train_df)
    return m

def evaluate_prophet(train_df: pd.DataFrame, val_df: pd.DataFrame):
    m = fit_prophet_model(train_df)

    # Create a dataframe of all dates up to the end of validation
    last_val_date = val_df["ds"].max()
    future = m.make_future_dataframe(periods=0, freq="MS")  # only train dates
    # For predictions, concat train+val horizon:
    future = pd.concat([train_df[["ds"]], val_df[["ds"]]]).drop_duplicates().sort_values("ds")

    forecast = m.predict(future)

    # Extract predictions only for validation window
    forecast_val = forecast[forecast["ds"].isin(val_df["ds"])]
    y_true = val_df["y"].values
    y_pred = forecast_val["yhat"].values

    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)

    print(
        f"[Prophet] Validation window: {val_df['ds'].min().date()} → {val_df['ds'].max().date()}"
    )
    print(f"[Prophet] Validation RMSE: {rmse:.2f}")
    print(f"[Prophet] Validation MAE : {mae:.2f}")

    return m, forecast, rmse, mae

prophet_model, prophet_forecast_full, prophet_rmse, prophet_mae = evaluate_prophet(
    train_df, val_df
)


13:34:07 - cmdstanpy - INFO - Chain [1] start processing
13:34:07 - cmdstanpy - INFO - Chain [1] done processing
13:34:07 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 


RuntimeError: Error during optimization! Command 'E:\Research_Final\component3_service\venv\Lib\site-packages\prophet\stan_model\prophet_model.bin random seed=81567 data file=C:\Users\ASUS\AppData\Local\Temp\tmp02u71jok\7wwcnufz.json init=C:\Users\ASUS\AppData\Local\Temp\tmp02u71jok\i9j9k084.json output file=C:\Users\ASUS\AppData\Local\Temp\tmp02u71jok\prophet_model51ytjk1v\prophet_model-20251123133407.csv method=optimize algorithm=newton iter=10000' failed: 

In [ ]:
from prophet import Prophet
import pandas as pd
import numpy as np
import math
from sklearn.metrics import mean_squared_error, mean_absolute_error

def fit_prophet_model(train_df: pd.DataFrame) -> Prophet:
    # Work on a copy
    df = train_df.copy()

    # Keep only required columns
    df = df[["ds", "y"]]

    # Ensure ds is datetime
    df["ds"] = pd.to_datetime(df["ds"])

    # Ensure y is numeric
    df["y"] = pd.to_numeric(df["y"], errors="coerce")

    # Drop NaNs / inf
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["ds", "y"])

    # Sort by date & drop duplicate dates
    df = df.sort_values("ds")
    df = df[~df["ds"].duplicated()]

    print(f"[Prophet] Training on {len(df)} rows from {df['ds'].min().date()} to {df['ds'].max().date()}")

    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode="additive",
    )

    # If optimization is touchy, you *can* force a different algorithm:
    # m.fit(df, algorithm="LBFGS")
    m.fit(df)
    return m


In [ ]:
def evaluate_prophet(train_df: pd.DataFrame, val_df: pd.DataFrame):
    # Fit on cleaned train_df
    m = fit_prophet_model(train_df)

    # Build future dates as union of train + val ds
    future = pd.concat(
        [train_df[["ds"]], val_df[["ds"]]],
        axis=0
    ).drop_duplicates().sort_values("ds")

    forecast = m.predict(future)

    # Extract predictions only for validation window
    forecast_val = forecast[forecast["ds"].isin(val_df["ds"])].sort_values("ds")
    val_df_sorted = val_df.sort_values("ds")

    y_true = val_df_sorted["y"].values
    y_pred = forecast_val["yhat"].values

    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)

    print(
        f"[Prophet] Validation window: {val_df_sorted['ds'].min().date()} → {val_df_sorted['ds'].max().date()}"
    )
    print(f"[Prophet] Validation RMSE: {rmse:.2f}")
    print(f"[Prophet] Validation MAE : {mae:.2f}")

    return m, forecast, rmse, mae


In [ ]:
prophet_model, prophet_forecast_full, prophet_rmse, prophet_mae = evaluate_prophet(
    train_df, val_df
)


13:37:03 - cmdstanpy - INFO - Chain [1] start processing
13:37:03 - cmdstanpy - INFO - Chain [1] done processing
13:37:03 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 


[Prophet] Training on 82 rows from 2018-01-01 to 2024-10-01


RuntimeError: Error during optimization! Command 'E:\Research_Final\component3_service\venv\Lib\site-packages\prophet\stan_model\prophet_model.bin random seed=6071 data file=C:\Users\ASUS\AppData\Local\Temp\tmp02u71jok\zy9ckz8o.json init=C:\Users\ASUS\AppData\Local\Temp\tmp02u71jok\3yuagn1e.json output file=C:\Users\ASUS\AppData\Local\Temp\tmp02u71jok\prophet_model3c7t6_kd\prophet_model-20251123133703.csv method=optimize algorithm=newton iter=10000' failed: 

In [ ]:
from prophet import Prophet
import pandas as pd
import numpy as np

print("Prophet sanity test...")

toy_df = pd.DataFrame({
    "ds": pd.date_range(start="2015-01-01", periods=36, freq="MS"),
    "y": np.random.randint(100, 1000, size=36)
})

print(toy_df.dtypes)
print(toy_df.head())

m_toy = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
)
m_toy.fit(toy_df)

print("✅ Toy Prophet model fitted successfully.")


13:42:43 - cmdstanpy - INFO - Chain [1] start processing


Prophet sanity test...
ds    datetime64[ns]
y              int32
dtype: object
          ds    y
0 2015-01-01  585
1 2015-02-01  217
2 2015-03-01  194
3 2015-04-01  748
4 2015-05-01  994


13:42:43 - cmdstanpy - INFO - Chain [1] done processing
13:42:43 - cmdstanpy - ERROR - Chain [1] error: code '3221225785' 


RuntimeError: Error during optimization! Command 'E:\Research_Final\component3_service\venv\Lib\site-packages\prophet\stan_model\prophet_model.bin random seed=58345 data file=C:\Users\ASUS\AppData\Local\Temp\tmp02u71jok\sgp6ur6d.json init=C:\Users\ASUS\AppData\Local\Temp\tmp02u71jok\xna3laa3.json output file=C:\Users\ASUS\AppData\Local\Temp\tmp02u71jok\prophet_modelppm1hco7\prophet_model-20251123134243.csv method=optimize algorithm=newton iter=10000' failed: 